In [ ]:
"""
build_erl_risk_summary.py

Rebuild ERL risk summary CSVs from ERL shapefiles.

Reads shapefiles in:
  F:\\2023_El_Rio_Lobo\\RSF_ERL\\field_1\\rsf_risk_exports
  F:\\2023_El_Rio_Lobo\\RSF_ERL\\field_2\\rsf_risk_exports
  F:\\2023_El_Rio_Lobo\\RSF_ERL\\field_3\\rsf_risk_exports

Assumes each shapefile has:
  - Date (date or date-like string)
  - field_id (field identifier)
  - risk_norm (or risk_norm_ or similar; alias is NOT used)
  - tmt_grp (treatment group; optional but expected)

Risk categories derived from risk_norm thresholds:
  - Low      : risk_norm <= 0.15
  - Moderate : 0.15 < risk_norm <= 0.40
  - High     : risk_norm > 0.40

Outputs:
1) ERL_2023_RiskType_Counts_Percent_AllFields.csv
   Columns: Field, Date, Category, Count, Total, Percent

2) ERL_2023_RiskType_Counts_Percent_AllFields_BYTMT.csv
   Columns: Field, Date, tmt_grp, Category, Count, Total, Percent

Run:
  python build_erl_risk_summary.py
"""

from __future__ import annotations

from pathlib import Path
from typing import Optional
import pandas as pd
import geopandas as gpd


# ----------------------------
# CONFIG
# ----------------------------
ROOT = Path(r"F:\2023_El_Rio_Lobo\RSF_ERL")

FIELD_DIRS = [
    ROOT / "field_1" / "rsf_risk_exports",
    ROOT / "field_2" / "rsf_risk_exports",
    ROOT / "field_3" / "rsf_risk_exports",
]

# Required fields (actual schema field names, not aliases)
DATE_COL = "Date"
FIELDID_COL = "field_id"

# risk_norm can vary after joins/exports; aliases do not matter in Python
RISK_NORM_CANDIDATES = [
    "risk_norm",
    "risk_norm_",
    "risk_norm_1",
    "risk_norm__",
    "risknorm",      # just in case
    "risk_norm2",    # just in case
]

# Treatment group field (you said it's called this)
TMT_GRP_CANDIDATES = [
    "tmt_grp",
    "TMT_GRP",
    "tmtgrp",
]

OUT_CSV = ROOT / "ERL_2023_RiskType_Counts_Percent_AllFields.csv"
OUT_CSV_BYTMT = ROOT / "ERL_2023_RiskType_Counts_Percent_AllFields_BYTMT.csv"


# ----------------------------
# Helpers
# ----------------------------
def pick_first_existing_col(columns: list[str], candidates: list[str]) -> Optional[str]:
    """Return first candidate found in columns (case-insensitive), else None."""
    col_map = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in col_map:
            return col_map[cand.lower()]
    return None


def parse_date(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce")


def normalize_field_id(field_id: object) -> str:
    """
    Standardize Field output.
    - If field_id is 1/2/3 -> "Field 1" etc.
    - If already "Field 1" -> keep
    """
    if field_id is None or (isinstance(field_id, float) and pd.isna(field_id)):
        return "Unknown"
    s = str(field_id).strip()
    if s.lower().startswith("field"):
        return s
    return f"Field {s}"


def classify_risk(r: float) -> str | pd.NA:
    """Apply your thresholds to risk_norm."""
    if pd.isna(r):
        return pd.NA
    if r <= 0.15:
        return "Low"
    elif r <= 0.40:
        return "Moderate"
    else:
        return "High"


def summarize_shapefile(shp: Path) -> tuple[pd.DataFrame, pd.DataFrame | None]:
    """
    Returns:
      - by_risk: Field, Date, Category, Count, Total, Percent
      - by_tmt : Field, Date, tmt_grp, Category, Count, Total, Percent
                (None if tmt_grp field not found)
    """
    gdf = gpd.read_file(shp)

    # Required base fields check
    missing = [c for c in (DATE_COL, FIELDID_COL) if c not in gdf.columns]
    if missing:
        raise KeyError(f"{shp.name}: Missing required fields {missing}")

    # Find risk_norm (real field name, not alias)
    risk_col = pick_first_existing_col(list(gdf.columns), RISK_NORM_CANDIDATES)
    if risk_col is None:
        raise KeyError(
            f"{shp.name}: No risk_norm field found. "
            f"Tried {RISK_NORM_CANDIDATES}. "
            f"Columns={list(gdf.columns)}"
        )

    # Find treatment group if present
    tmt_col = pick_first_existing_col(list(gdf.columns), TMT_GRP_CANDIDATES)

    df = pd.DataFrame({
        "Field": gdf[FIELDID_COL].map(normalize_field_id).astype("string"),
        "Date": parse_date(gdf[DATE_COL]),
        "risk_norm": pd.to_numeric(gdf[risk_col], errors="coerce"),
    })

    df["Category"] = df["risk_norm"].map(classify_risk)

    # Drop bad records
    df = df.dropna(subset=["Date", "Category"])

    # ----------------------------
    # Summary: by risk (Field x Date)
    # ----------------------------
    by_risk = (
        df.groupby(["Field", "Date", "Category"], dropna=False)
          .size()
          .reset_index(name="Count")
    )
    by_risk["Total"] = by_risk.groupby(["Field", "Date"])["Count"].transform("sum")
    by_risk["Percent"] = (by_risk["Count"] / by_risk["Total"]) * 100.0

    # ----------------------------
    # Summary: by risk + treatment group (Field x Date x tmt_grp)
    # ----------------------------
    by_tmt = None
    if tmt_col is not None:
        df2 = df.copy()
        df2["tmt_grp"] = gdf[tmt_col].astype("string").fillna("")

        # Optional: treat blanks as "Standard (350)" if you want
        # (comment out if blanks are meaningful)
        df2.loc[df2["tmt_grp"].str.strip() == "", "tmt_grp"] = "Standard (350)"

        by_tmt = (
            df2.groupby(["Field", "Date", "tmt_grp", "Category"], dropna=False)
               .size()
               .reset_index(name="Count")
        )
        by_tmt["Total"] = by_tmt.groupby(["Field", "Date", "tmt_grp"])["Count"].transform("sum")
        by_tmt["Percent"] = (by_tmt["Count"] / by_tmt["Total"]) * 100.0

    return by_risk, by_tmt


def main() -> int:
    shp_files: list[Path] = []
    for d in FIELD_DIRS:
        if not d.exists():
            print(f"[WARN] Folder not found: {d}")
            continue
        shp_files.extend(sorted(d.glob("*.shp")))

    if not shp_files:
        print("[ERROR] No shapefiles found. Check ROOT / FIELD_DIRS.")
        return 1

    all_by_risk: list[pd.DataFrame] = []
    all_by_tmt: list[pd.DataFrame] = []
    saw_any_tmt = False

    print(f"Found {len(shp_files)} shapefiles.\n")

    for shp in shp_files:
        try:
            by_risk, by_tmt = summarize_shapefile(shp)
            all_by_risk.append(by_risk)
            if by_tmt is not None:
                saw_any_tmt = True
                all_by_tmt.append(by_tmt)
            print(f"[OK] {shp.name}")
        except Exception as e:
            print(f"[WARN] Skipping {shp.name}: {e}")

    if not all_by_risk:
        print("\n[ERROR] No summaries produced (all files failed).")
        return 1

    out = pd.concat(all_by_risk, ignore_index=True)

    # Category order (nice sorting)
    cat_order = pd.CategoricalDtype(["High", "Moderate", "Low"], ordered=True)
    out["Category"] = out["Category"].astype(cat_order)
    out = out.sort_values(["Field", "Date", "Category"])

    out.to_csv(OUT_CSV, index=False)
    print(f"\nWrote: {OUT_CSV}")

    if saw_any_tmt and all_by_tmt:
        out2 = pd.concat(all_by_tmt, ignore_index=True)
        out2["Category"] = out2["Category"].astype(cat_order)
        out2 = out2.sort_values(["Field", "Date", "tmt_grp", "Category"])
        out2.to_csv(OUT_CSV_BYTMT, index=False)
        print(f"Wrote: {OUT_CSV_BYTMT}")
    else:
        print("No tmt_grp field found in any shapefile; skipping BYTMT output.")

    return 0


if __name__ == "__main__":
    code = main()
    print(f"Done (exit code {code}).")




Found 18 shapefiles.

[OK] ERL_Field_1_20230421_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field_1_20230515_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field_1_20230522_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field_1_20230601_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field_1_20230610_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field_1_20230620_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field2_20230421_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field2_20230515_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field2_20230522_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field2_20230601_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field2_20230610_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field2_20230620_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field3_20230421_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field3_20230515_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field3_20230522_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field3_20230601_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field3_20230610_HexMetrics_WithRisk_TRT.shp
[OK] ERL_Field3_20230620_HexMetrics_WithRisk_TRT.shp

Wrote: F:\2023_El

In [16]:
"""
make_risk_frames.py

Creates stacked-area PNG frames that look like the example:
- Moderate (bottom) + High (top)
- y-axis: % of At-Risk Plants
- count labels centered within each stacked band
- one image per Field per frame date, showing data up to that date

Input:
  ERL_2023_RiskType_Counts_Percent_AllFields.csv

Output folder (created if needed):
  <ROOT>/risk_frames/
"""

from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


# ----------------------------
# CONFIG
# ----------------------------
ROOT = Path(r"F:\2023_El_Rio_Lobo\RSF_ERL")
IN_CSV = ROOT / "ERL_2023_RiskType_Counts_Percent_AllFields.csv"
OUT_DIR = ROOT / "risk_frames"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Risk categories to plot and their stacking order (bottom -> top)
STACK_ORDER = ["Moderate", "High"]

# Y axis: keep consistent across plots like your example
Y_MIN, Y_MAX = 0, 30
Y_TICKS = np.arange(0, 31, 5)

# Alpha of filled areas
ALPHA = 0.40

# If you want to use a specific set of frame dates, put them here (YYYY-MM-DD strings)
# Leave as None to use all dates found in the CSV.
START_DATE = pd.Timestamp("2023-05-15")
END_DATE   = pd.Timestamp("2023-06-10")

FRAME_DATES = ["2023-05-15", "2023-05-22", "2023-06-01", "2023-06-10"]


# ----------------------------
# Helpers
# ----------------------------
def prep_data(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, parse_dates=["Date"])

    # Normalize Field labels like "Field 1.0" -> "Field 1"
    df["Field"] = (
        df["Field"].astype(str)
          .str.replace(r"\.0$", "", regex=True)
          .str.strip()
    )

    # Keep only categories we need
    df = df[df["Category"].isin(STACK_ORDER)].copy()

    # Clean types
    df["Percent"] = pd.to_numeric(df["Percent"], errors="coerce")
    df["Count"] = pd.to_numeric(df["Count"], errors="coerce")
    df = df.dropna(subset=["Field", "Date", "Category", "Percent", "Count"])

    return df


def pivot_for_stack(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns (pct_wide, cnt_wide) indexed by Date, columns=Category in STACK_ORDER.
    Missing combos become 0.
    """
    pct = (
        df.pivot_table(index="Date", columns="Category", values="Percent", aggfunc="sum")
          .fillna(0.0)
    )
    cnt = (
        df.pivot_table(index="Date", columns="Category", values="Count", aggfunc="sum")
          .fillna(0.0)
    )

    # enforce order and ensure all columns exist
    for c in STACK_ORDER:
        if c not in pct.columns:
            pct[c] = 0.0
        if c not in cnt.columns:
            cnt[c] = 0.0

    pct = pct[STACK_ORDER].sort_index()
    cnt = cnt[STACK_ORDER].sort_index()

    return pct, cnt

from datetime import timedelta

def add_count_labels(ax, dates, pct_wide, cnt_wide):
    if len(dates) == 0:
        return

    left_date = dates.min()
    right_date = dates.max()

    cum = pct_wide.cumsum(axis=1)
    base = cum.shift(axis=1).fillna(0.0)
    y_mid = base + pct_wide / 2.0

    y_min_label = 0.6  # floor above x-axis
    tiny_threshold = 1.0  # percent; defines a "tiny" band

    for i, cat in enumerate(STACK_ORDER):
        for x, y, c, band_height in zip(
            dates,
            y_mid[cat].values,
            cnt_wide[cat].values,
            pct_wide[cat].values,
        ):
            if not c or c <= 0:
                continue

            # X edge nudging
            x_plot = x
            ha = "center"
            if x == left_date:
                x_plot = x + timedelta(days=1)
                ha = "left"
            elif x == right_date:
                x_plot = x - timedelta(days=1)
                ha = "right"

            # Base y position
            y_plot = max(y, y_min_label)

            # If band is very thin, add separation between categories
            if band_height < tiny_threshold:
                y_plot += i * 1  # stagger vertically

            ax.text(x_plot, y_plot, f"{int(c)}", ha=ha, va="center", fontsize=8)





def plot_frame(field_name: str, pct_wide: pd.DataFrame, cnt_wide: pd.DataFrame,
               frame_date: pd.Timestamp, xlim_dates: list[pd.Timestamp]):
    """
    Plot one frame (data up to frame_date).
    """
    # Subset up to frame_date
    pct_sub = pct_wide[pct_wide.index <= frame_date]
    cnt_sub = cnt_wide[cnt_wide.index <= frame_date]

    # Ensure we still have the same x-axis ticks/limits
    dates_sub = pct_sub.index

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(6.4, 4.8))  # similar aspect to your example

    # Stack areas (matplotlib picks default colors; we keep it simple)
    ax.stackplot(
        dates_sub,
        [pct_sub[c].values for c in STACK_ORDER],
        labels=STACK_ORDER,
        alpha=ALPHA,
        linewidth=0.5,
    )

    # Labels
    ax.set_xlabel("Date")
    ax.set_ylabel("% of At-Risk Plants")

    # Y scale (percent units already 0–100, so show 0–30 with % sign)
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_yticks(Y_TICKS)
    ax.set_yticklabels([f"{int(t)}%" for t in Y_TICKS])

    # X axis formatting: use the master frame dates as ticks (like your example)
    ax.set_xlim(START_DATE, END_DATE)

    ax.set_xticks(xlim_dates)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

    # Legend (upper-left-ish)
    ax.legend(title="Risk", loc="upper left")

    # Grid similar vibe
    ax.grid(True, alpha=0.3)

    # Count labels
    add_count_labels(ax, dates_sub, pct_sub, cnt_sub)

    # Title optional (comment out if you want exact match with no title)
    # ax.set_title(f"{field_name} (through {frame_date:%Y-%m-%d})")

    fig.tight_layout()

    # Save
    fname_date = frame_date.strftime("%Y%m%d")
    safe_field = "".join(ch if ch.isalnum() or ch in ("_", "-") else "_" for ch in field_name)
    out_path = OUT_DIR / f"Field_{safe_field}_Date_{fname_date}_Risk.png"
    fig.savefig(out_path, dpi=300)
    plt.close(fig)


def main():
    df = prep_data(IN_CSV)
    df = df[(df["Date"] >= START_DATE) & (df["Date"] <= END_DATE)].copy()

    fields = sorted(df["Field"].unique())

    # Decide frame dates
    if FRAME_DATES is None:
        frame_dates = sorted(df["Date"].dropna().unique())
        frame_dates = [pd.Timestamp(d) for d in frame_dates]
    else:
        frame_dates = [pd.Timestamp(d) for d in FRAME_DATES]

    # Restrict frames to date window
    frame_dates = [d for d in frame_dates if START_DATE <= d <= END_DATE]

    if not frame_dates:
        raise RuntimeError("No frame dates found in selected window.")

    for field in fields:
        df_f = df[df["Field"] == field].copy()

        pct_wide, cnt_wide = pivot_for_stack(df_f)

        if pct_wide.empty:
            print(f"[SKIP] {field}: no High/Moderate data.")
            continue

        for fd in frame_dates:
            plot_frame(field, pct_wide, cnt_wide, fd, frame_dates)

        print(f"[OK] Wrote frames for {field}")



if __name__ == "__main__":
    main()


[OK] Wrote frames for Field 1
[OK] Wrote frames for Field 2
[OK] Wrote frames for Field 3
